# q4: products
which products sell? (revenue, units, orders)
uses purchase items only: view_item and add_to_cart events carry ~11 different products each, so product-level views and carts are unreliable (see q4b in sql/06_products.sql).


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
with purchased_items as (
    select distinct
        ecommerce.transaction_id,
        item.item_name,
        item.item_variant,
        item.quantity,
        item.item_revenue_in_usd as item_revenue
    from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`,
    unnest(items) as item
    where event_name = 'purchase'
    and ecommerce.transaction_id != '(not set)'
)
select
    item_name,
    count(distinct transaction_id) as orders,
    sum(quantity) as units,
    sum(item_revenue) as revenue
from purchased_items
group by item_name
order by revenue desc
"""
products = client.query(sql).to_dataframe()
products


,item_name,orders,units,revenue
0,Google Zip Hoodie F/C,187,208,10488.0
1,Google Crewneck Sweatshirt Navy,196,217,9878.0
2,Google Badge Heavyweight Pullover Black,150,176,8490.0
3,Google Men's Tech Fleece Grey,87,107,7886.0
4,Super G Unisex Joggers,221,253,7856.0
...,...,...,...,...
385,Google Women's Ringer Tee,1,1,14.0
386,Google Kirkland Campus Lapel Pin,2,2,6.0
387,Android Flex Journal White,2,3,6.0
388,Google Separating Keyring,1,1,6.0


In [3]:
products["revenue_share"] = (products["revenue"] / products["revenue"].sum()).round(4)
products["avg_price"] = (products["revenue"] / products["units"]).round(2)
print(len(products), products["revenue"].sum())
products.head(20)


390 306714.0


,item_name,orders,units,revenue,revenue_share,avg_price
0,Google Zip Hoodie F/C,187,208,10488.0,0.0342,50.42
1,Google Crewneck Sweatshirt Navy,196,217,9878.0,0.0322,45.52
2,Google Badge Heavyweight Pullover Black,150,176,8490.0,0.0277,48.24
3,Google Men's Tech Fleece Grey,87,107,7886.0,0.0257,73.7
4,Super G Unisex Joggers,221,253,7856.0,0.0256,31.05
5,Google Crewneck Sweatshirt Green,146,155,7095.0,0.0231,45.77
6,Google Men's Tech Fleece Vest Charcoal,62,79,5234.0,0.0171,66.25
7,Google Men's Puff Jacket Black,52,55,5221.0,0.0170,94.93
8,Google Sherpa Zip Hoodie Charcoal,78,89,4953.0,0.0161,55.65
9,Google Canteen Bottle Black,130,250,4930.0,0.0161,19.72


In [4]:
products.to_csv(output_dir / "q4_products.csv", index=False)